# FTEC5660 Reproducibility: ACE on the Formula task (Google Colab)

Run the cells top to bottom. A CPU runtime is enough, since all inference goes through the DeepSeek API.

**Before you start**
1. Click the key icon (Secrets) in the left sidebar → add a secret named `DEEPSEEK_API_KEY` holding your key, and switch on *Notebook access*.
2. Results are copied to Google Drive (`MyDrive/ftec5660_ace_results`) after every stage, so a disconnect does not lose finished work.
3. Keep this tab open while `train` runs (about 1 hour). Free Colab disconnects idle sessions.

## 1. Get the code
Option A (recommended): you already pushed the repo to GitHub → clone it.  
Option B: upload `ace-ftec5660-repro.zip` with the Files panel, then run the unzip line instead.

In [ ]:
%cd /content
# Option A: clone your fork (replace the URL)
!git clone -b repro-ftec5660 https://github.com/<your-username>/ace.git
# Option B: uploaded zip
# !unzip -q -o /content/ace-ftec5660-repro.zip -d /content
%cd /content/ace
!git log --oneline | head -8

## 2. Install dependencies and set the key

In [ ]:
!pip -q install openai tiktoken python-dotenv
import os
from google.colab import userdata
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
os.environ["PY"] = "python"          # use Colab's python instead of uv
os.environ["MODEL"] = "deepseek-chat"
os.environ["TRIALS"] = "3"
os.environ["MAX_TRAIN"] = "60"       # lower to 30 if you are short on time
os.environ["MAX_VAL"] = "100"
os.environ["EVAL_STEPS"] = "20"
os.environ["WORKERS"] = "10"
!python --version

## 3. Save results to Google Drive after each stage

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/ftec5660_ace_results"
os.makedirs(DRIVE, exist_ok=True)
os.makedirs("/content/ace/logs", exist_ok=True)

def backup():
    for d in ["results", "logs"]:
        src = f"/content/ace/{d}"
        if os.path.isdir(src):
            shutil.copytree(src, f"{DRIVE}/{d}", dirs_exist_ok=True)
    print("backed up to", DRIVE)

## 4. Smoke test (5 questions, about 1 minute)

In [ ]:
!bash scripts/run_repro.sh smoke > logs/smoke.log 2>&1; tail -n 12 logs/smoke.log

## 5. R1 baseline: empty playbook, 3 × 200 test questions

In [ ]:
!bash scripts/run_repro.sh baseline > logs/baseline.log 2>&1; grep -E "Test Accuracy|Final Accuracy" logs/baseline.log
backup()

## 6. R2 ACE offline training (longest step)
Runs in the background so you can check progress with the next cell. Re-run the progress cell every few minutes; wait until it prints `RUN COMPLETE`.

In [ ]:
!nohup bash scripts/run_repro.sh train > logs/train.log 2>&1 &
print("started")

In [ ]:
# progress: re-run this cell
!grep -oE "Step [0-9]+/[0-9]+" logs/train.log | tail -1; grep -E "Initial Test Accuracy|New best|Final Test Accuracy|RUN COMPLETE|Error|Traceback" logs/train.log | tail -8

In [ ]:
backup()  # run once training has finished

## 7. R2: evaluate the learned playbook 2 more times

In [ ]:
!bash scripts/run_repro.sh ace_eval > logs/ace_eval.log 2>&1; grep -E "Test Accuracy" logs/ace_eval.log
backup()

## 8. Summary and modification (re-scoring, no extra API cost)
Set the DeepSeek prices (USD per 1M tokens, from the DeepSeek pricing page) to get a cost estimate, or leave them empty.

In [ ]:
os.environ["PRICE_IN"] = ""    # e.g. input price per 1M tokens
os.environ["PRICE_OUT"] = ""   # e.g. output price per 1M tokens
for k in ["PRICE_IN", "PRICE_OUT"]:
    if not os.environ[k]: os.environ.pop(k)
!bash scripts/run_repro.sh summary > logs/summary.log 2>&1
backup()
from IPython.display import Markdown, display
display(Markdown(open("results/summary.md").read()))

## 9. (Optional) ACE trained with the robust grader as feedback
Only if you have time and budget left; then re-run section 8.

In [ ]:
# !bash scripts/run_repro.sh robust_train > logs/robust_train.log 2>&1; tail -n 5 logs/robust_train.log
# backup()

## 10. Download
`results/summary.md` holds every number the report needs. Download it and the best playbook:

In [ ]:
from google.colab import files
import glob
files.download("results/summary.md")
files.download(sorted(glob.glob("results/ace_train/ace_run_*/best_playbook.txt"))[-1])